In [1]:
import datasets
from datasets import load_dataset

dataset = load_dataset("dair-ai/emotion", split="train")

print("Dataset LOADED")
print(dataset[0])

Dataset LOADED
{'text': 'i didnt feel humiliated', 'label': 0}


In [2]:
from transformers import AutoTokenizer, AutoModel

model_checkpnt = "sentence-transformers/multi-qa-mpnet-base-dot-v1"

tokenizer = AutoTokenizer.from_pretrained(model_checkpnt)

model = AutoModel.from_pretrained(model_checkpnt)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/multi-qa-mpnet-base-dot-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
import torch 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Directing model calculations to device: {device}")

model.to(device)

Directing model calculations to device: cuda


MPNetModel(
  (embeddings): MPNetEmbeddings(
    (word_embeddings): Embedding(30527, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): MPNetEncoder(
    (layer): ModuleList(
      (0-11): 12 x MPNetLayer(
        (attention): MPNetAttention(
          (attn): MPNetSelfAttention(
            (q): Linear(in_features=768, out_features=768, bias=True)
            (k): Linear(in_features=768, out_features=768, bias=True)
            (v): Linear(in_features=768, out_features=768, bias=True)
            (o): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (intermediate): MPNetIntermediate(
          (dense): Linear(in_

In [4]:
def cls_pooling(model_output):
    return model_output.last_hidden_state[:, 0]

def embed_and_pool_batch(batch):
    encoded_input = tokenizer(batch['text'],padding= True,truncation=True,return_tensors="pt")

    encoded_input = {k: v.to(device) for k, v in encoded_input.items()}

    with torch.no_grad():
        outputs = model(**encoded_input)

    cls_vectors = cls_pooling(outputs).cpu().numpy()

    return {"embeddings": cls_vectors}
##we will stream the data from the hf cloud so my computer ram doesnt fill up doing it from the disk localy
stream_dataset = load_dataset("dair-ai/emotion", split="train", streaming=True)

##we will ship 32 batch every round, better for optimaization.If you have good hardware there is no need to use this, since your device can handle embedding the whole dataset at a signle turn
embedded_stream = stream_dataset.map(embed_and_pool_batch, batched=True, batch_size=32)

local_dataset = dataset.from_generator(lambda: iter(embedded_stream))



Generating train split: 0 examples [00:00, ? examples/s]

In [5]:
print(local_dataset.column_names)

['text', 'label', 'embeddings']


In [6]:
local_dataset.add_faiss_index(column="embeddings")

  0%|          | 0/16 [00:00<?, ?it/s]

Dataset({
    features: ['text', 'label', 'embeddings'],
    num_rows: 16000
})

In [18]:
def search_engine(query,k = 3):
    input = tokenizer(query,padding=True,truncation=True,return_tensors="pt").to(device)

    with torch.no_grad():
        output = model(**input)

    query_cls = cls_pooling(output).cpu().numpy()
    scores, samples = local_dataset.get_nearest_examples("embeddings", query_cls, k=k)

    for i in range(len(scores)):
        print(f"Match #{i+1} | Distance Score: {scores[i]:.4f}")
        print(f"Tweet: {samples['text'][i]}")
        print("-" * 65)

search_engine("im feeling playful already", k=10)

Match #1 | Distance Score: 0.0000
Tweet: im feeling playful already
-----------------------------------------------------------------
Match #2 | Distance Score: 8.8767
Tweet: im feeling playful and humorous
-----------------------------------------------------------------
Match #3 | Distance Score: 10.6831
Tweet: i am hoping i am still feeling playful in a few days
-----------------------------------------------------------------
Match #4 | Distance Score: 14.0093
Tweet: i was feeling playful
-----------------------------------------------------------------
Match #5 | Distance Score: 14.7729
Tweet: i was feeling rather playful last night as well
-----------------------------------------------------------------
Match #6 | Distance Score: 15.7311
Tweet: i started to mess around something must have distracted me cause now im feeling playful
-----------------------------------------------------------------
Match #7 | Distance Score: 17.7192
Tweet: i feel playful today a href http www
-----